# Frontier Inference Systems

দুটি demo:

1. MHA বনাম GQA বনাম MQA বনাম MLA-এর KV-cache memory, Lesson 3 Sec 4-এর প্রকৃত cache-size ফর্মুলা (`2 * batch * seq_len * num_kv_heads * d_k * num_layers * bytes_per_value`) ব্যবহার করে, যেখানে MLA-এর ক্ষেত্রে `num_kv_heads * d_k` term-টি একটি একক ছোট `d_latent` দিয়ে প্রতিস্থাপিত। একটি সুনির্দিষ্ট, স্পষ্টভাবে-চিহ্নিত **দৃষ্টান্তমূলক** (ILLUSTRATIVE) model configuration-এর জন্য গণনা করা (কোনো নির্দিষ্ট বাস্তব প্রকাশিত model-এর হুবহু config **নয়**), context length-এর একটি sweep জুড়ে, চারটি attention variant-এর মধ্যে প্রকৃত গণনাকৃত অনুপাত-সহ।
2. Request-এর একটি stream-এর Monte Carlo simulation, যাদের prompt-গুলো "prefix"-এর একটি ছোট, Zipfian-skewed সেট ভাগ করে (কয়েকটি খুব সাধারণ, অনেকগুলো বিরল), একটি নির্দিষ্ট সংখ্যক replica জুড়ে route করা, যার প্রতিটিতে একটি সীমিত-capacity LRU prefix cache আছে। আমরা **প্রকৃত** simulated cache-hit rate মাপি (a) round-robin routing-এর অধীনে, যা cache state পুরোপুরি উপেক্ষা করে, বনাম (b) একটি সরল cache-aware routing নিয়মের অধীনে, যা একটি request-কে সেই replica-তে পাঠায় যে সবচেয়ে সম্প্রতি তার হুবহু prefix serve করেছে, কেবল cache miss হলে round-robin-এ ফিরে যায়।

Runtime: যেকোনো machine-এ এক সেকেন্ডেরও অনেক কম (বিশুদ্ধ arithmetic এবং কয়েক হাজার-iteration-এর Monte Carlo loop, কোনো model training নেই)।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান।

In [ ]:
import random
from collections import OrderedDict

random.seed(0)

## 1. KV-cache memory: MHA বনাম GQA বনাম MQA বনাম MLA

এই সেকশন একটি দৃষ্টান্তমূলক বড় dense decoder-only model configuration সংজ্ঞায়িত করে, Lesson 3 Sec 4-এর KV-cache ফর্মুলা বাস্তবায়ন করে, এবং তিনটি context length-এ (4K, 32K, 128K) চারটি attention variant-এর cache আকার GB-তে গণনা করে। MLA-এর জন্য `num_kv_heads * d_k` term-টি একটি একক `d_latent`-এ পরিণত হয় — প্রতি token প্রতি layer-এ একটিমাত্র cached vector। শেষে সবচেয়ে দীর্ঘ context-এ প্রকৃত অনুপাতগুলো ছাপা হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 1. KV-cache memory: MHA বনাম GQA বনাম MQA বনাম MLA
#
#    Lesson 3 Sec 4-এর ফর্মুলা:
#        KV cache bytes = 2 * batch * seq_len * num_kv_heads * d_k
#                          * num_layers * bytes_per_value
#    MLA `num_kv_heads * d_k` term-টিকে একটি একক latent dimension
#    `d_latent` দিয়ে প্রতিস্থাপন করে, প্রতি (গ্রুপের) head-এ আলাদা K/V
#    আদৌ cache না করে attention-এর সময় শেখা up-projection-এর মাধ্যমে সেই
#    একটি latent vector থেকে per-head K/V পুনর্গঠন করে।
# ---------------------------------------------------------------------------

# একটি সুনির্দিষ্ট, দৃষ্টান্তমূলক (ILLUSTRATIVE) model configuration -- কোনো
# নির্দিষ্ট বাস্তব প্রকাশিত model-এর হুবহু সংখ্যা নয়, শুধু একটি বড় dense
# decoder-only model-এর বাস্তবসম্মত order of magnitude, স্পষ্টভাবে বলা যাতে
# নিচের arithmetic পুরোপুরি পুনরুৎপাদনযোগ্য হয়।
NUM_LAYERS = 60
NUM_HEADS = 96
D_K = 128                       # per-head Key/Value dimension
GQA_NUM_KV_HEADS = NUM_HEADS // 8   # প্রতি ৮টি head-এর গ্রুপে একটি shared KV pair
MQA_NUM_KV_HEADS = 1                # সব head জুড়ে একটি shared KV pair
D_LATENT = 64                       # MLA-এর প্রতি token/layer-এ একক cached latent dim
                                     # (ইচ্ছাকৃতভাবে একটি GQA গ্রুপের নিজস্ব
                                     # d_k=128-এর চেয়ে ছোট, Lesson 3 Sec 4-এর ফর্মুলা অনুযায়ী)
BYTES_PER_VALUE = 2                  # bf16/fp16 storage
BATCH = 1                            # একটি একক sequence-এর জন্য cache আকার


def kv_cache_bytes(seq_len, num_kv_heads_or_latent, num_layers=NUM_LAYERS,
                    d_k=None, batch=BATCH, bytes_per_value=BYTES_PER_VALUE):
    """Lesson 3 Sec 4-এর ফর্মুলা। MHA/GQA/MQA-এর জন্য num_kv_heads_or_latent
    = num_kv_heads এবং d_k = D_K দিন (term-টি হলো num_kv_heads * d_k)। MLA-এর
    জন্য num_kv_heads_or_latent = d_latent এবং d_k = 1 দিন (term-টি কেবল
    d_latent -- প্রতি token প্রতি layer-এ একটি একক cached vector, প্রতি head-এ নয়)।"""
    per_token_per_layer_dim = num_kv_heads_or_latent * (d_k if d_k is not None else 1)
    return 2 * batch * seq_len * per_token_per_layer_dim * num_layers * bytes_per_value


def gb(num_bytes):
    return num_bytes / (1024 ** 3)


def kv_cache_memory_demo():
    print("=" * 78)
    print("1. KV CACHE MEMORY: MHA vs. GQA vs. MQA vs. MLA")
    print("=" * 78)
    print("Illustrative model config (NOT a specific real published model):")
    print(f"  num_layers={NUM_LAYERS}, num_heads={NUM_HEADS}, d_k={D_K}, "
          f"bytes_per_value={BYTES_PER_VALUE} (bf16), batch={BATCH}")
    print(f"  GQA num_kv_heads = num_heads/8 = {GQA_NUM_KV_HEADS}")
    print(f"  MQA num_kv_heads = {MQA_NUM_KV_HEADS}")
    print(f"  MLA d_latent = {D_LATENT}  (< one GQA group's own d_k={D_K})\n")

    context_lengths = [4_096, 32_768, 131_072]
    variants = [
        ("MHA (num_kv_heads=num_heads)", NUM_HEADS, D_K),
        ("GQA (num_kv_heads=num_heads/8)", GQA_NUM_KV_HEADS, D_K),
        ("MQA (num_kv_heads=1)", MQA_NUM_KV_HEADS, D_K),
        ("MLA (single d_latent per token)", D_LATENT, 1),
    ]

    header = f"{'context length':>16}" + "".join(f"{name.split(' ')[0]:>14}" for name, _, _ in variants)
    print(header + "   (GB)")
    results = {}
    for seq_len in context_lengths:
        row = f"{seq_len:>16,}"
        results[seq_len] = {}
        for name, num_kv_or_latent, d_k in variants:
            b = kv_cache_bytes(seq_len, num_kv_or_latent, d_k=d_k)
            results[seq_len][name] = b
            row += f"{gb(b):>14.3f}"
        print(row)

    print("\nReal computed ratios at the longest context (131,072 tokens):")
    longest = context_lengths[-1]
    mha_bytes = results[longest]["MHA (num_kv_heads=num_heads)"]
    for name, _, _ in variants:
        ratio = mha_bytes / results[longest][name]
        print(f"  MHA uses {ratio:6.1f}x the cache memory of {name}"
              if name != "MHA (num_kv_heads=num_heads)"
              else f"  MHA is the {ratio:.1f}x baseline")

    mla_bytes = results[longest]["MLA (single d_latent per token)"]
    gqa_bytes = results[longest]["GQA (num_kv_heads=num_heads/8)"]
    print(f"\n-> At {longest:,} tokens of context, MLA uses "
          f"{mha_bytes / mla_bytes:.1f}x LESS cache memory than plain MHA, and "
          f"{gqa_bytes / mla_bytes:.1f}x less than GQA -- GQA/MQA shrink the cache by sharing")
    print("   identical K/V across a group of heads (fewer distinct num_kv_heads); MLA shrinks it")
    print("   further by caching a single small latent vector per token instead of per-head K/V at")
    print("   all, reconstructing full multi-head K/V from that latent via up-projection at attention")
    print("   time -- more aggressive memory savings, at the cost of that extra per-step up-projection")
    print("   compute that GQA/MQA's directly-cached, reused K/V never has to pay.")


kv_cache_memory_demo()

## 2. Cache-aware fleet routing বনাম round-robin: প্রকৃত simulated hit rate

এখানে Zipfian-skewed জনপ্রিয়তা-সহ prefix-এর একটি ছোট pool (বাস্তব workload-এর shared system prompt / few-shot template / দীর্ঘ document-এর stand-in) থেকে request-এর একটি stream তৈরি হয়। প্রতিটি replica-র একটি সীমিত-capacity LRU prefix cache (`ReplicaCache`) আছে। আমরা একই stream-এ দুটি routing policy তুলনা করি: (a) round-robin, যা cache state উপেক্ষা করে, এবং (b) cache-aware, যা সেই replica-কে অগ্রাধিকার দেয় যে সবচেয়ে সম্প্রতি হুবহু সেই prefix serve করেছে — এবং প্রকৃত hit-rate ব্যবধান ছাপি।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Cache-aware fleet routing বনাম round-robin: প্রকৃত simulated hit rate।
#
#    "prefix"-এর একটি ছোট pool Zipfian-skewed জনপ্রিয়তা-সহ টানা হয়
#    (কয়েকটি prefix খুব সাধারণ, বেশিরভাগ বিরল) -- একটি বাস্তব workload-এর
#    shared system prompt / few-shot template / দীর্ঘ document-এর stand-in।
#    প্রতিটি replica-র বর্তমানে ধরে রাখা prefix-গুলোর একটি সীমিত-capacity
#    LRU cache আছে। আমরা তুলনা করি:
#      (a) round-robin routing: request i যায় replica (i % num_replicas)-এ,
#          কোন replica-তে এই prefix cached আছে তা পুরোপুরি উপেক্ষা করে।
#      (b) cache-aware routing: সেই replica-তে route করো যে সবচেয়ে সম্প্রতি
#          হুবহু এই prefix serve করেছে, যদি সেই replica-তে এটি এখনো cached
#          থাকে; অন্যথায় round-robin-এ ফিরে যাও।
# ---------------------------------------------------------------------------

NUM_PREFIXES = 24
ZIPF_SKEW = 1.3          # বেশি -> কয়েকটি জনপ্রিয় prefix-এর দিকে বেশি skewed
NUM_REPLICAS = 4
CACHE_CAPACITY = 6       # প্রতিটি replica-র cache কতগুলো স্বতন্ত্র prefix ধরে রাখতে পারে
NUM_REQUESTS = 4_000


def zipfian_weights(num_items, skew):
    return [1.0 / ((rank + 1) ** skew) for rank in range(num_items)]


def make_prefix_stream(num_requests, num_prefixes, skew):
    prefixes = [f"prefix_{i}" for i in range(num_prefixes)]
    weights = zipfian_weights(num_prefixes, skew)
    # random.choices প্রকৃত weighted random sampling করে, প্রতি request-এ একটি prefix
    # (request-এর অনন্য per-user suffix cache routing/hit logic-এর জন্য অপ্রাসঙ্গিক,
    # তাই এটি বাদ দেওয়া হয়েছে -- এখানে শুধু shared prefix-এর পরিচয়ই গুরুত্বপূর্ণ)।
    return random.choices(prefixes, weights=weights, k=num_requests)


class ReplicaCache:
    """একটি replica বর্তমানে যে prefix-গুলো ধরে রেখেছে তার একটি ক্ষুদ্র LRU cache।"""

    def __init__(self, capacity):
        self.capacity = capacity
        self._store = OrderedDict()

    def has(self, key):
        return key in self._store

    def touch(self, key):
        """রেকর্ড করো যে `key` এইমাত্র এই replica serve করেছে (এটি আগে থেকে
        cached থাকুক বা না থাকুক), এটিকে most-recently-used হিসেবে চিহ্নিত করো
        এবং cache এখন capacity ছাড়িয়ে গেলে least-recently-used entry-টি evict করো।"""
        if key in self._store:
            self._store.move_to_end(key)
        else:
            self._store[key] = True
            if len(self._store) > self.capacity:
                self._store.popitem(last=False)


def simulate_round_robin(prefix_stream, num_replicas, capacity):
    caches = [ReplicaCache(capacity) for _ in range(num_replicas)]
    hits = 0
    for i, prefix in enumerate(prefix_stream):
        replica = i % num_replicas
        if caches[replica].has(prefix):
            hits += 1
        caches[replica].touch(prefix)
    return hits / len(prefix_stream)


def simulate_cache_aware(prefix_stream, num_replicas, capacity):
    caches = [ReplicaCache(capacity) for _ in range(num_replicas)]
    last_served_by = {}   # prefix -> যে replica id সবচেয়ে সম্প্রতি এটি serve করেছে
    rr_counter = 0
    hits = 0
    for prefix in prefix_stream:
        preferred = last_served_by.get(prefix)
        if preferred is not None and caches[preferred].has(prefix):
            replica = preferred
            hits += 1
        else:
            # cache miss (অথবা আগে কখনো দেখা হয়নি): replica-গুলোর মধ্যে
            # round-robin-এ ফিরে যাও, ঠিক naive policy যেমন করত।
            replica = rr_counter % num_replicas
            rr_counter += 1
        caches[replica].touch(prefix)
        last_served_by[prefix] = replica
    return hits / len(prefix_stream)


def cache_aware_routing_demo():
    print("\n" + "=" * 78)
    print("2. CACHE-AWARE FLEET ROUTING vs. ROUND-ROBIN: SIMULATED HIT RATES")
    print("=" * 78)
    print(f"{NUM_REQUESTS:,} requests, {NUM_PREFIXES} distinct shared prefixes "
          f"(Zipfian skew={ZIPF_SKEW}), {NUM_REPLICAS} replicas, "
          f"each replica's prefix cache holds {CACHE_CAPACITY} prefixes.\n")

    weights = zipfian_weights(NUM_PREFIXES, ZIPF_SKEW)
    total_w = sum(weights)
    print("Prefix popularity (share of traffic), most popular first:")
    for rank in range(5):
        print(f"  prefix_{rank}: {weights[rank] / total_w * 100:5.1f}% of requests")
    print(f"  ... plus {NUM_PREFIXES - 5} rarer prefixes making up the rest\n")

    prefix_stream = make_prefix_stream(NUM_REQUESTS, NUM_PREFIXES, ZIPF_SKEW)

    rr_hit_rate = simulate_round_robin(prefix_stream, NUM_REPLICAS, CACHE_CAPACITY)
    ca_hit_rate = simulate_cache_aware(prefix_stream, NUM_REPLICAS, CACHE_CAPACITY)

    print(f"{'routing policy':>28}{'cache hit rate':>18}")
    print(f"{'round-robin (cache-blind)':>28}{rr_hit_rate * 100:>17.1f}%")
    print(f"{'cache-aware':>28}{ca_hit_rate * 100:>17.1f}%")

    improvement = ca_hit_rate / rr_hit_rate if rr_hit_rate > 0 else float("inf")
    print(f"\n-> Cache-aware routing achieves a {improvement:.1f}x higher hit rate than round-robin on "
          f"the SAME simulated request stream, replica count, and cache capacity -- the only")
    print("   difference is whether the router uses cache-topology information (which replica last")
    print("   served this exact prefix) when making its routing decision. Round-robin scatters even")
    print("   a very popular prefix's requests evenly across all replicas, so any one replica sees")
    print("   it only a fraction of the time and its LRU cache entry is frequently evicted between")
    print("   visits; cache-aware routing keeps a popular prefix's requests concentrated on whichever")
    print("   replica already has it hot, avoiding redundant prefill exactly as Lesson 6 Sec 2's")
    print("   single-server prefix caching does -- just applied at the level of the whole fleet.")


cache_aware_routing_demo()

## সবগুলো demo একসাথে

`main()` উপরের দুটি demo (KV-cache memory তুলনা এবং cache-aware routing simulation) একসাথে এক টানে চালায়। প্রতিটি demo ইতিমধ্যে নিজের cell-এ চলেছে, তাই demo দুবার না চালাতে নিচের call-টি comment করে রাখা হয়েছে।

In [ ]:
def main():
    kv_cache_memory_demo()
    cache_aware_routing_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন